In [1]:
import sys
import time
from IPython.display import clear_output

# Assuming SerialPrinter is inside a local file named printer.py
# If it's in the same folder, this will import it directly:
try:
    from printer import SerialPrinter
except ImportError:
    print("Ensure your SerialPrinter class code is saved in a file named 'printer.py' in this directory.")

# Initialize and connect to the printer
printer = SerialPrinter()
printer.connect()

Connecting to GRBL on 0...
Connecting to GRBL on 1...
Connecting to GRBL on 2...
Connecting to GRBL on 3...
Connecting to GRBL on 4...
Connecting to GRBL on 5...
Waiting...
Connected successfully.
start
Marlin 2.1.2.5
echo: Last Updated: 2024-11-18 | Author: (none, default config)
echo: Compiled: Jun 10 2026
echo: Free Memory: 5619  PlannerBufferBytes: 1136
echo:Hardcoded Default Settings Loaded
echo:; Linear Units:
echo:  G21 ; (mm)
echo:; Temperature Units:
echo:  M149 C ; Units in Celsius
echo:; Filament settings (Disabled):
echo:  M200 S0 D1.75
echo:; Steps per unit:
echo:  M92 X80.00 Y80.00 Z400.00 E500.00
echo:; Max feedrates (units/s):
echo:  M203 X300.00 Y300.00 Z5.00 E25.00
echo:; Max Acceleration (units/s2):
echo:  M201 X3000.00 Y3000.00 Z100.00 E10000.00
echo:; Acceleration (units/s2) (P<print-accel> R<retract-accel> T<travel-accel>):
echo:  M204 P3000.00 R3000.00 T3000.00
echo:; Advanced (B<min_segment_time_us> S<min_feedrate> T<min_travel_feedrate> J<junc_dev>):
echo:  M20

In [2]:
def query_limit_switches(printer_instance):
    """
    Sends the Marlin endstop status command 'M119' and parses the response.
    """
    if not printer_instance.connection or not printer_instance.connection.is_open:
        return "Not Connected"
    
    # M119 is the standard Marlin command to report endstop status
    cmd = "M119\r\n"
    printer_instance.connection.write(cmd.encode('utf-8'))
    
    # Give the printer a split second to reply, then read the buffer
    time.sleep(0.1)
    
    response_lines = []
    while printer_instance.connection.in_waiting > 0:
        line = printer_instance.connection.readline().decode('utf-8').strip()
        if line:
            response_lines.append(line)
            
    # Marlin typically responds with lines like:
    # x_min: open
    # y_min: TRIGGERED
    # z_min: open
    
    triggered_switches = []
    for line in response_lines:
        if "triggered" in line.lower():
            # Extract the switch name (e.g., "x_min" from "x_min: TRIGGERED")
            switch_name = line.split(":")[0].strip()
            triggered_switches.append(switch_name)
            
    if triggered_switches:
        return f"TRIGGERED: {', '.join(triggered_switches)}"
        
    return "All Switches Open (Clear)"

In [3]:
# 1. Enable Hard Limits ($21=1)
print(printer.send_command("$21=1"))

# 2. Enable Homing Cycle ($22=1) - Required for switch checks
print(printer.send_command("$22=1"))

# 3. CRITICAL: Tell GRBL to include Pin Status in the '?' status report ($10=2 or $10=3)
# In newer GRBL versions, bits 1 and 2 of $10 control position and buffer/pin data distribution.
print(printer.send_command("$10=3"))

echo:Unknown command: "$21=1"
ok
echo:Unknown command: "$21=1"
ok
echo:Unknown command: "$22=1"
ok
echo:Unknown command: "$22=1"
ok
echo:Unknown command: "$10=3"
ok
echo:Unknown command: "$10=3"
ok


In [4]:
print(printer.send_command("$$"))

echo:Unknown command: "$$"
ok
echo:Unknown command: "$$"
ok


In [5]:
print("Starting live switch monitor... Press Stop/Interrupt to exit.")
time.sleep(1)

try:
    while True:
        status = query_limit_switches(printer)
        
        # Clear the notebook output block for a clean UI feel
        clear_output(wait=True)
        
        print("=========================================")
        print("       GRBL LIMIT SWITCH MONITOR        ")
        print("=========================================")
        print(f" Current Status: {status}")
        print("-----------------------------------------")
        print(" Physically press your limit switches now to test them.")
        
        time.sleep(0.2) # Query ~5 times per second

except KeyboardInterrupt:
    print("\nMonitoring stopped by user.")

       GRBL LIMIT SWITCH MONITOR        
 Current Status: TRIGGERED: x_min, y_min, z_min
-----------------------------------------
 Physically press your limit switches now to test them.

Monitoring stopped by user.
